## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-28-00-50-43 +0000,startribune,Live: Gophers take early lead over Wisconsin i...,https://www.startribune.com/minnesota-wisconsi...
1,2026-11-28-00-46-42 +0000,nyt,NBC Pulls an Ad That Some Described as Antisem...,https://www.nytimes.com/2026/11/27/business/me...
2,2026-11-28-00-43-25 +0000,nyt,Supreme Court Rulings Cheered by Republicans W...,https://www.nytimes.com/2026/11/27/us/politics...
3,2026-11-28-00-35-52 +0000,nyt,"Thanksgiving Storm Leaves More Than 400,000 Wi...",https://www.nytimes.com/2026/11/27/us/thanksgi...
4,2026-11-28-00-32-32 +0000,startribune,Live: Caledonia leads defending champ Jackson ...,https://www.startribune.com/minnesota-high-sch...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2715/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
240,new,19
163,trump,18
23,thanksgiving,15
45,out,13
249,black,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
277,2026-11-27-04-06-00 +0000,wsj,New Zealand’s election ended in a deadlock aft...,https://www.wsj.com/politics/elections/new-zea...,64
54,2026-11-27-21-52-33 +0000,nypost,Manic shoppers camp out early for Black Friday...,https://nypost.com/2026/11/27/us-news/californ...,56
86,2026-11-27-20-26-11 +0000,wapo,Trump officials to speed president’s projects ...,https://www.washingtonpost.com/politics/2026/1...,56
29,2026-11-27-23-04-41 +0000,chitri,Shoppers turn out for Black Friday deals as ho...,https://www.chicagotribune.com/2026/11/27/blac...,56
96,2026-11-27-19-55-08 +0000,chitri,Black Friday brings shoppers out in force in A...,https://www.chicagotribune.com/2026/11/27/blac...,54


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
277,64,2026-11-27-04-06-00 +0000,wsj,New Zealand’s election ended in a deadlock aft...,https://www.wsj.com/politics/elections/new-zea...
29,56,2026-11-27-23-04-41 +0000,chitri,Shoppers turn out for Black Friday deals as ho...,https://www.chicagotribune.com/2026/11/27/blac...
196,51,2026-11-27-12-24-03 +0000,chitri,Ukraine president urges US to pressure Vladimi...,https://www.chicagotribune.com/2026/11/27/ukra...
139,49,2026-11-27-17-16-23 +0000,nypost,NBC faces backlash for airing anti-Israel ad d...,https://nypost.com/2026/11/27/media/nbc-faces-...
143,47,2026-11-27-16-53-46 +0000,bbc,Canada's top court largely upholds Quebec ban ...,https://www.bbc.co.uk/news/articles/cq3dgpy9le...
147,37,2026-11-27-16-35-08 +0000,nypost,Real cause of death of Deb Fischer’s husband r...,https://nypost.com/2026/11/27/us-news/real-cau...
24,34,2026-11-27-23-14-37 +0000,nypost,DHS accuses California of endangering ICE offi...,https://nypost.com/2026/11/27/us-news/dhs-accu...
253,32,2026-11-27-08-32-00 +0000,wsj,European Stock Indexes Rise as Tech Rally Extends,https://www.wsj.com/finance/stocks/nikkei-rise...
130,31,2026-11-27-17-51-29 +0000,wapo,Hundreds of thousands on East Coast remain wit...,https://www.washingtonpost.com/nation/2026/11/...
204,30,2026-11-27-11-53-57 +0000,nypost,Iranian state media releases AI video threaten...,https://nypost.com/2026/11/27/world-news/irani...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
